In [2]:
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("table_tableau08.csv")

In [5]:
df

,Political affiliation/Appartenance politique,N.L. Valid Votes/Votes valides T.-N.-L.,P.E.I. Valid Votes/Votes valides Î.-P.-É.,N.S. Valid Votes/Votes valides N.-É.,N.B. Valid Votes/Votes valides N.-B.,Que. Valid Votes/Votes valides Qc,Ont. Valid Votes/Votes valides Ont.,Man. Valid Votes/Votes valides Man.,Sask. Valid Votes/Votes valides Sask.,Alta. Valid Votes/Votes valides Alb.,B.C. Valid Votes/Votes valides C.-B.,Y.T. Valid Votes/Votes valides Yn,N.W.T. Valid Votes/Votes valides T.N.-O.,Nun. Valid Votes/Votes valides Nt
0,Animal Protection Party of Canada/Le Parti pou...,170,0,0,0,0,723,213,0,0,195,0,0,0
1,Bloc Québécois/Bloc Québécois,0,0,0,0,1236349,0,0,0,0,0,0,0,0
2,Canadian Future Party/Parti Avenir Canadien,0,0,169,345,0,872,0,568,1079,90,0,0,0
3,Centrist Party of Canada/Parti Centriste du Ca...,0,0,0,44,60,3103,0,0,107,0,0,0,0
4,Christian Heritage Party of Canada/Parti de l'...,0,0,0,0,181,5193,0,0,3604,1087,0,0,0
5,Communist Party of Canada/Parti communiste du ...,98,0,0,146,572,1707,428,0,865,869,0,0,0
6,Conservative Party of Canada/Parti conservateu...,110259,35846,204051,190429,1040992,3324929,297510,362495,1442094,1088655,8719,5513,1992
7,Green Party of Canada/Le Parti Vert du Canada,299,2170,5448,7982,40795,85907,4579,2869,8944,79255,474,170,0
8,Liberal Party of Canada/Parti libéral du Canada,150458,56011,332354,250659,1904206,3730652,261113,149397,631929,1105033,12009,8855,2812
9,Libertarian Party of Canada/Parti Libertarien ...,0,0,320,776,0,2484,0,226,623,1132,0,0,0


How many seats does each province elect?

In [ ]:
seat_allocation = {
    # Provinces
    "Ontario": 122,
    "Quebec": 77,
    "British Columbia": 43,
    "Alberta": 37,
    "Manitoba": 14,
    "Saskatchewan": 14,
    "Nova Scotia": 11,
    "New Brunswick": 10,
    "Newfoundland and Labrador": 7,
    "Prince Edward Island": 4
    # Territories use FPTP in practice as they just have one seat each
}

In [8]:
# Source - https://stackoverflow.com/a/51754373
# Posted by George Liu, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-16, License - CC BY-SA 4.0

df = df.reset_index()
df_melt = df.melt(
    id_vars="Political affiliation/Appartenance politique",
    var_name="Province",
    value_name="Votes"
)

In [9]:
df_melt

,Political affiliation/Appartenance politique,Province,Votes
0,Animal Protection Party of Canada/Le Parti pou...,index,0
1,Bloc Québécois/Bloc Québécois,index,1
2,Canadian Future Party/Parti Avenir Canadien,index,2
3,Centrist Party of Canada/Parti Centriste du Ca...,index,3
4,Christian Heritage Party of Canada/Parti de l'...,index,4
...,...,...,...
247,Parti Rhinocéros Party/Parti Rhinocéros Party,Nun. Valid Votes/Votes valides Nt,0
248,People's Party of Canada/Parti populaire du Ca...,Nun. Valid Votes/Votes valides Nt,0
249,United Party of Canada (UP)/Parti Uni du Canad...,Nun. Valid Votes/Votes valides Nt,0
250,Independent/Indépendant(e),Nun. Valid Votes/Votes valides Nt,0


In [13]:
NS = df_melt[df_melt["Province"] == "N.S. Valid Votes/Votes valides N.-É."]
PE = df_melt[df_melt["Province"] == "P.E.I. Valid Votes/Votes valides Î.-P.-É."]
ON = df_melt[df_melt["Province"] == "Ont. Valid Votes/Votes valides Ont."]
QC = df_melt[df_melt["Province"] == "Que. Valid Votes/Votes valides Qc"]
SK = df_melt[df_melt["Province"] == "Sask. Valid Votes/Votes valides Sask."]
MB = df_melt[df_melt["Province"] == "Man. Valid Votes/Votes valides Man."]
NB = df_melt[df_melt["Province"] == "Sask. Valid Votes/Votes valides Sask."]
NL = df_melt[df_melt["Province"] == "N.L. Valid Votes/Votes valides T.-N.-L."]
AB = df_melt[df_melt["Province"] == "Alta. Valid Votes/Votes valides Alb."]
BC = df_melt[df_melt["Province"] == "B.C. Valid Votes/Votes valides C.-B."]

In [18]:
ON

,Political affiliation/Appartenance politique,Province,Votes
108,Animal Protection Party of Canada/Le Parti pou...,Ont. Valid Votes/Votes valides Ont.,723
109,Bloc Québécois/Bloc Québécois,Ont. Valid Votes/Votes valides Ont.,0
110,Canadian Future Party/Parti Avenir Canadien,Ont. Valid Votes/Votes valides Ont.,872
111,Centrist Party of Canada/Parti Centriste du Ca...,Ont. Valid Votes/Votes valides Ont.,3103
112,Christian Heritage Party of Canada/Parti de l'...,Ont. Valid Votes/Votes valides Ont.,5193
113,Communist Party of Canada/Parti communiste du ...,Ont. Valid Votes/Votes valides Ont.,1707
114,Conservative Party of Canada/Parti conservateu...,Ont. Valid Votes/Votes valides Ont.,3324929
115,Green Party of Canada/Le Parti Vert du Canada,Ont. Valid Votes/Votes valides Ont.,85907
116,Liberal Party of Canada/Parti libéral du Canada,Ont. Valid Votes/Votes valides Ont.,3730652
117,Libertarian Party of Canada/Parti Libertarien ...,Ont. Valid Votes/Votes valides Ont.,2484


In [14]:
def dhondt(province, seats):
    votes = dict(zip(
        province["Political affiliation/Appartenance politique"],
        province["Votes"]
    ))
    
    allocation = {party: 0 for party in votes}
    quotients = []

    for party, vote in votes.items():
        for i in range(1, seats + 1):
            quotients.append((party, vote / i))

    quotients.sort(key=lambda x: x[1], reverse=True)

    for i in range(seats):
        party = quotients[i][0]
        allocation[party] += 1

    return allocation

In [20]:
on = dhondt(ON, 122)

In [21]:
print(on)

{'Animal Protection Party of Canada/Le Parti pour la Protection des Animaux du Canada': 0, 'Bloc Québécois/Bloc Québécois': 0, 'Canadian Future Party/Parti Avenir Canadien': 0, 'Centrist Party of Canada/Parti Centriste du Canada': 0, "Christian Heritage Party of Canada/Parti de l'Héritage Chrétien du Canada": 0, 'Communist Party of Canada/Parti communiste du Canada': 0, 'Conservative Party of Canada/Parti conservateur du Canada': 54, 'Green Party of Canada/Le Parti Vert du Canada': 1, 'Liberal Party of Canada/Parti libéral du Canada': 61, 'Libertarian Party of Canada/Parti Libertarien du Canada': 0, 'Marijuana Party/Parti Marijuana': 0, 'Marxist-Leninist Party of Canada/Parti Marxiste-Léniniste du Canada': 0, 'New Democratic Party/Nouveau Parti démocratique': 6, 'Parti Rhinocéros Party/Parti Rhinocéros Party': 0, "People's Party of Canada/Parti populaire du Canada": 0, 'United Party of Canada (UP)/Parti Uni du Canada (UP)': 0, 'Independent/Indépendant(e)': 0, 'No Affiliation/Aucune app